In [1]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error

from data import CaseVFL

In [33]:
N = 1000
feature_count = 100
parties = 5

party_size = feature_count // parties
assert party_size * parties == feature_count

rng = np.random.default_rng(42)
case = CaseVFL(
    rng = rng,
    n = N,
    events = feature_count,
    output_arity = 50,
    sensors = parties,
    sensor_arity = party_size,
    noise_std = 0.01
)

X_train, Y_train = case.generate_data()
X_test, Y_test = case.generate_data()
X_train_agg = np.block([x.T for x in X_train])
X_test_agg = np.block([x.T for x in X_test])

# Baseline

In [34]:
model = Ridge(alpha=0.5).fit(X_train_agg, Y_train)
Y_train_pred = model.predict(X_train_agg)
print(f"Train Error: {mean_squared_error(Y_train_pred, Y_train)}")
Y_test_pred = model.predict(X_test_agg)
print(f"Test Error: {mean_squared_error(Y_test_pred, Y_test)}")

Train Error: 9.110656181217162e-05
Test Error: 0.00011959060203509258


# VFL

In [61]:
weights = [rng.uniform(size=(party_size)) for _ in range(parties)]
for iteration in range(300000):
    Y_hat = np.sum([(weights[i] @ X_train[i]) for i in range(parties)], axis=0)
    eta = 1 / 10000
    for i in range(parties):
        weights[i] -= (X_train[i] @ (Y_hat - Y_train)) * 2 / N * eta

In [62]:
Y_train_hat = np.sum([(weights[i] @ X_train[i]) for i in range(parties)], axis=0)
print("Train", mean_squared_error(Y_train_hat, Y_train))
Y_test_hat = np.sum([(weights[i] @ X_test[i]) for i in range(parties)], axis=0)
print("Test", mean_squared_error(Y_test_hat, Y_test))

Train 0.00011999820096028784
Test 0.00015616766513884705


# Gradient Inversion

In [171]:
T = 100000
X_atk, Y_atk = case.generate_data()
weights = [rng.uniform(size=(party_size)) for _ in range(parties)]
gradients = []

for iteration in range(T):
    grad = [(weights[i] @ X_train[i]) for i in range(parties)]
    Y_hat = np.sum(grad, axis=0)
    if iteration % 100 == 0:
        gradients.append(grad)
    eta = 1 / 10000
    for i in range(parties):
        weights[i] -= (X_train[i] @ (Y_hat - Y_train)) * 2 / N * eta

In [172]:
Y_atk_hat = np.sum([(weights[i] @ X_atk[i]) for i in range(parties)], axis=0)
print("Train", mean_squared_error(Y_atk_hat, Y_atk))

Train 0.0024580711786308276


In [187]:
recovered_features = []
recovered_weights = []
for p in range(parties):
    G = np.vstack([gradients[t][p] for t in range(T // 100)]).T
    U, S, Vt = np.linalg.svd(G, full_matrices=False)

    # restrict to relevant values
    r = np.sum(S > 1e-10)
    U_r = U[:,:r]
    S_r = S[:r]
    V_r = Vt[:r,:].T

    weights_r = (np.diag(S_r) @ V_r.T)[:, -1]

    recovered_features.append(U_r)
    recovered_weights.append(weights_r)

In [188]:
Y_hat = sum(recovered_features[p] @ recovered_weights[p] for p in range(parties))
mean_squared_error(Y_hat, Y_atk)

2918.9304915546477

In [192]:
"""
Federated GD: https://dl.acm.org/doi/pdf/10.1145/1281192.1281275
Cocktail Party: https://arxiv.org/pdf/2209.05578
CAFE: https://arxiv.org/pdf/2110.15122 (new gradient inversion tech)
Survey: https://arxiv.org/pdf/2206.07284
"""

'\nFederated GD: https://dl.acm.org/doi/pdf/10.1145/1281192.1281275\nCocktail Party: https://arxiv.org/pdf/2209.05578\nCAFE: https://arxiv.org/pdf/2110.15122 (new gradient inversion tech)\nSurvey: https://arxiv.org/pdf/2206.07284\n'